# NEB Workflow — Script Generator + Analysis

This notebook is a **script generator**. Running cells 1–3 writes two files to disk:
- `neb_run.py` — standalone Python orchestrator (Phases A–D)
- `neb_run.sh` — SLURM submission script configured by notebook variables

Submitting `neb_run.sh` runs the full workflow on the cluster unattended.  
Cells 4+ are run **locally after the cluster jobs complete** to inspect and visualise results.

---

**Phase A — Slab preparation** *(multigpu partition)*  
&nbsp;&nbsp; Build FCC(111) Hastelloy N slab → CG minimise → NVT surface relaxation → enumerate 171 adsorption sites

**Phase B — Adsorption energies** *(multigpu partition)*  
&nbsp;&nbsp; H₂\* (molecular) adsorption for 171 sites → IS pool (intact H₂\* candidates)  
&nbsp;&nbsp; H\* (atomic) adsorption for 171 sites → FS pool (E_ads < 0)

**Phase C — NEB enumeration + job generation**  
&nbsp;&nbsp; FS pair enumeration (separation + graph-distance filters)  
&nbsp;&nbsp; Chemical-environment deduplication (IS sites + FS pairs)  
&nbsp;&nbsp; Cross-product → proximity + label-key filter → ~914 unique pathways  
&nbsp;&nbsp; Per-job: IS structure, FS raw structure, FS-min script, ASE NEB script, SLURM scripts  
&nbsp;&nbsp; Array scripts: `run_fsmin_array.sh` (GPU) + `run_neb_array.sh` (CPU)

**Phase D — Submit and wait**  
&nbsp;&nbsp; Submit FS-min GPU array → wait → submit NEB CPU array → wait

**Phase 4 — Local analysis & visualisation** *(run in this notebook)*  
&nbsp;&nbsp; 4a. Load `neb_barrier.txt` files → summary DataFrame  
&nbsp;&nbsp; 4b. Barrier heatmap (mean E_a per IS × FS-pair chemical environment)  
&nbsp;&nbsp; 4c. MEP overlay (all paths; top-10 lowest barriers highlighted)

## Cells 1–2: Imports & configuration

Edit `BULK_MIN_PATH`, `E_H2_GAS`, SLURM configs, and NEB parameters here before generating scripts.

In [1]:
import os
import sys

# Project root = the directory containing models/. Found by walking up from the
# working directory rather than counting parents, so this keeps working wherever
# the notebook is moved to. (These notebooks now live two levels deeper than
# calculation/, where the old fixed-depth version silently put the wrong path
# on sys.path.)
_root = os.path.abspath(os.getcwd())
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'models')):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print(f'project root: {_root}')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
from models.config import (
    LAMMPS_CMD, MACE_MODEL_LAMMPS, MACE_MODEL_ASE, KOKKOS_FLAGS,
    PAIR_STYLE, PAIR_SUFFIX,
    build_element_tables,
    SLURM_DEFAULTS, BASE_DIR,
    N_REPLICAS, SPRING_CONST, NEB_FTOL,
)
from models.create_slurm import submit_slurm_job
from models.neb_workflow import calculate_ref_adsorbate_energy
from models.structure import build_element_tables_from_structure

# ── User-editable config ──────────────────────────────────────────────────────
# BASE_DIR = '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel'
WORK_DIR = os.path.join(BASE_DIR, 'calculation')

# ── Structure input ───────────────────────────────────────────────────────────
BULK_MIN_PATH = os.path.join(BASE_DIR, 'structures/bulk_min.lammps')

# ── Output directories ────────────────────────────────────────────────────────
SLAB_DIR = os.path.join(WORK_DIR, 'slabs')
ADS_DIR  = os.path.join(WORK_DIR, 'adsorption')
NEB_DIR  = os.path.join(WORK_DIR, 'neb')

# ── Dry-run mode — set False when ready to submit to cluster ─────────────────
dry_run = True

# ── Slab geometry ─────────────────────────────────────────────────────────────
MILLER     = (1, 1, 1)
LAYERS     = 12
VACUUM     = 15.0    # Å
LAT_REPEAT = (5, 6)

# ── Surface relaxation (Phase A — slab prep)
Z_FREEZE_CUTOFF  = None     # Å — None = auto: freeze bottom 1/3 of slab thickness
SURF_TIMESTEP_PS = 0.0005   # ps — NVT timestep for surface relaxation

# ── NEB pair-selection filters ────────────────────────────────────────────────
SEP_MIN        = 2.5   # Å  — min H*–H* XY separation
SEP_MAX        = 5.0   # Å  — max H*–H* XY separation
GRAPH_DIST_MIN = 2     # min surface-graph hops between FS sites
PROX_CUTOFF    = 3.0   # Å  — max IS centroid ↔ FS midpoint XY distance

# ── NEB parameters ────────────────────────────────────────────────────────────
N_IMAGES     = N_REPLICAS    # 9 intermediate images (from config)
SPRING_CONST = SPRING_CONST  # 1.0 eV/Å²
NEB_FTOL     = NEB_FTOL      # eV/Å force convergence
H_HEIGHT     = 1.5            # Å — H above surface in FS raw structure

# ── SLURM: GPU partition (slab relax, adsorption FS-min) ─────────────────────
GPU_SLURM_CFG = dict(SLURM_DEFAULTS, partition='multigpu', time='04:00:00')

# ── SLURM: CPU partition (ASE NEB jobs) ──────────────────────────────────────
NEB_SLURM_CFG = dict(SLURM_DEFAULTS, partition='short',
                     gpu=None, cpus_per_task=9, time='12:00:00')

# ── SLURM: CPU partition (vibrational frequency jobs — Phase E) ───────────────
VIB_SLURM_CFG = dict(SLURM_DEFAULTS, partition='short',
                     gpu=None, cpus_per_task=8,  time='06:00:00')

# ── SLURM: Orchestrator (neb_run.sh) ─────────────────────────────────────────
ORCH_JOB_NAME      = 'neb_orch'
ORCH_PARTITION     = 'west'
ORCH_CPUS_PER_TASK = 4
ORCH_TIME          = None    # set to e.g. '48:00:00' to enable wall-time limit
ORCH_LD_PATHS      = GPU_SLURM_CFG['ld_paths']
ORCH_OPENMPI_VER   = GPU_SLURM_CFG['openmpi_ver']
ORCH_CUDA_VERSION  = GPU_SLURM_CFG['cuda_version']
ORCH_CONDA_ENV     = GPU_SLURM_CFG['conda_env']

# ── Element table ── auto-detected from BULK_MIN_PATH's own structure, instead
# of a fixed hardcoded list -- works for any metal/alloy.
_E2T, _T2E, _MASSES, _ELEM_STR, _ = build_element_tables_from_structure(BULK_MIN_PATH)

# ── H₂ reference energy — computed with the same MACE model as the pipeline ──
# H2 is a synthetic vacuum-box molecule, not read from a structure file --
# this builds the minimal table directly from what's physically present (2 H
# atoms), verified to give identical results to a padded table (see
# calculation/tools/verify_padded_species_mace.py).
# dry_run=True: writes LAMMPS + SLURM scripts only; returns None (neb_run.py computes at runtime)
# dry_run=False: submits job, blocks until done, caches result to ADS_DIR/ref_energies/h2_ref_energy.json
_E2T_H, _T2E_H, _MASSES_H, _ELEM_STR_H = build_element_tables(['H'])
E_H2_GAS = calculate_ref_adsorbate_energy(
    adsorbate    = 'H2',
    outdir       = os.path.join(ADS_DIR, 'ref_energies'),
    pair_style   = PAIR_STYLE,
    mace_model   = MACE_MODEL_LAMMPS,
    pair_suffix  = PAIR_SUFFIX,
    elem_str     = _ELEM_STR_H,
    e2t          = _E2T_H,
    masses       = _MASSES_H,
    lammps_cmd   = LAMMPS_CMD,
    kokkos_flags = KOKKOS_FLAGS,
    slurm_opts   = GPU_SLURM_CFG,
    dry_run      = dry_run,
)

print('Config loaded.')
print(f'  WORK_DIR       : {WORK_DIR}')
print(f'  BULK_MIN_PATH  : {BULK_MIN_PATH}')
print(f'  E_H2_GAS       : {E_H2_GAS} eV  (None = not yet computed; neb_run.py auto-computes)')
print(f'  SLAB_DIR       : {SLAB_DIR}')
print(f'  ADS_DIR        : {ADS_DIR}')
print(f'  NEB_DIR        : {NEB_DIR}')
print(f'  N_IMAGES       : {N_IMAGES}')
print(f'  SPRING_CONST   : {SPRING_CONST} eV/Å²')
print(f'  NEB_FTOL       : {NEB_FTOL} eV/Å')
print(f'  GPU_PARTITION  : {GPU_SLURM_CFG["partition"]}')
print(f'  NEB_PARTITION  : {NEB_SLURM_CFG["partition"]}')
print(f'  VIB_PARTITION  : {VIB_SLURM_CFG["partition"]}')
print(f'  ORCH_PARTITION : {ORCH_PARTITION}')

## Cell 3: Generate `neb_run.py`

Builds the full Phase A→B→C→D orchestrator script as a Python file.  
All config values above are injected as literals into the generated file header.

In [3]:
from models.neb_workflow import write_neb_run_script

out_py = write_neb_run_script(
    bulk_min_path=BULK_MIN_PATH,
    work_dir=WORK_DIR,
    e_h2_gas=E_H2_GAS,
    slab_dir=SLAB_DIR,
    ads_dir=ADS_DIR,
    neb_dir=NEB_DIR,
    miller=MILLER,
    layers=LAYERS,
    vacuum=VACUUM,
    lat_repeat=LAT_REPEAT,
    z_freeze_cutoff=Z_FREEZE_CUTOFF,
    surf_timestep=SURF_TIMESTEP_PS,
    sep_min=SEP_MIN,
    sep_max=SEP_MAX,
    graph_dist_min=GRAPH_DIST_MIN,
    prox_cutoff=PROX_CUTOFF,
    n_images=N_IMAGES,
    spring_const=SPRING_CONST,
    neb_ftol=NEB_FTOL,
    h_height=H_HEIGHT,
    elem_str=_ELEM_STR,
    e2t=_E2T,
    masses=_MASSES,
    gpu_slurm_cfg=GPU_SLURM_CFG,
    neb_slurm_cfg=NEB_SLURM_CFG,
    vib_slurm_cfg=VIB_SLURM_CFG,
    out_py=os.path.join(os.getcwd(), 'neb_run.py'),
)
print(f'Written: {out_py}')

Written: /Users/akinyemi.az/Desktop/PhD_Folder/research/MHI/MD/Molecular_Dynamics/MHI_Nickel/calculation/neb_run.py


## Cell 4: Generate `neb_run.sh` and submit

Writes the SLURM orchestrator script and optionally submits it.  
Set `dry_run=False` when ready to submit to the cluster.

In [4]:
from models.neb_workflow import write_neb_orchestrator_sh

out_sh = write_neb_orchestrator_sh(
    orch_job_name=ORCH_JOB_NAME,
    orch_partition=ORCH_PARTITION,
    orch_cpus_per_task=ORCH_CPUS_PER_TASK,
    orch_time=ORCH_TIME,
    orch_openmpi_ver=ORCH_OPENMPI_VER,
    orch_cuda_version=ORCH_CUDA_VERSION,
    orch_conda_env=ORCH_CONDA_ENV,
    orch_ld_paths=ORCH_LD_PATHS,
    out_py=out_py,
    out_sh=os.path.join(os.getcwd(), 'neb_run.sh'),
)
print(f'Written: {out_sh}')

# ── Preview first 20 lines ────────────────────────────────────────────────────
with open(out_sh) as fh:
    for line in fh.readlines()[:20]:
        print(line, end='')

# ── Submit ────────────────────────────────────────────────────────────────────
# Set dry_run=False when ready to submit to the cluster.
job_id = submit_slurm_job(out_sh, dry_run=True)
print(f'\nOrchestrator job: {job_id}')

Written: /Users/akinyemi.az/Desktop/PhD_Folder/research/MHI/MD/Molecular_Dynamics/MHI_Nickel/calculation/neb_run.sh
#!/bin/bash
#SBATCH --job-name=neb_orch
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=4

#SBATCH --partition=west
#SBATCH --time=48:00:00
#SBATCH --output=/Users/akinyemi.az/Desktop/PhD_Folder/research/MHI/MD/Molecular_Dynamics/MHI_Nickel/calculation/neb_run_%j.out


module load OpenMPI/4.1.6
module load cuda/12.3.0
source ~/miniforge3/etc/profile.d/conda.sh
conda activate mace-lammps
export LD_LIBRARY_PATH=/shared/EL9/explorer/cuda/12.3.0/lib64/stubs:$LD_LIBRARY_PATH
export LD_LIBRARY_PATH=/projects/westgroup/akinyemi.az/mace_lammps/lammps/build-mliap:$LD_LIBRARY_PATH
export LD_LIBRARY_PATH=/home/akinyemi.az/miniforge3/envs/mace-lammps/lib:$LD_LIBRARY_PATH

cd /Users/akinyemi.az/Desktop/PhD_Folder/research/MHI/MD/Molecular_Dynamics/MHI_Nickel/calculation

[dry-run] sbatch /Users/akinyemi.az/Desktop/PhD_Folder/research/MHI/MD/Molecular_Dynamics/MHI_Nickel/calculation/neb_run

---
## Phase 4: Local analysis (run after cluster jobs complete)

### 4a. Load results — `neb_barrier.txt` → summary DataFrame

In [5]:
from models.neb_workflow import load_neb_results

barriers = load_neb_results(NEB_DIR)
if not barriers.empty:
    display(barriers[['is_label', 'fs_label1', 'fs_label2', 'E_abs', 'E_des',
                       'delta_E', 'converged', 'n_grouped']].head(15))

TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

### 4b. Barrier heatmap — mean E_a per IS × FS-pair chemical environment

In [ ]:
# Retired. These plotted an in-memory DataFrame keyed on environment
# labels (fs_label1/2), but the persisted artifact ranked_barriers.json
# is keyed on site ids (fs_site1/2), so they raised KeyError on real
# data. models/plots/neb_mep_plot.py reads the artifact directly and
# produces the per-stage overlays and the full pathway figure:
#
#   python models/plots/neb_mep_plot.py --pattern '<stem>'
#
# or let post-processing do it for you:
#
#   python models/plots/postprocess.py --stem <stem>


### 4c. MEP overlay — all NEB paths, top-10 lowest barriers highlighted

In [ ]:
# Retired. These plotted an in-memory DataFrame keyed on environment
# labels (fs_label1/2), but the persisted artifact ranked_barriers.json
# is keyed on site ids (fs_site1/2), so they raised KeyError on real
# data. models/plots/neb_mep_plot.py reads the artifact directly and
# produces the per-stage overlays and the full pathway figure:
#
#   python models/plots/neb_mep_plot.py --pattern '<stem>'
#
# or let post-processing do it for you:
#
#   python models/plots/postprocess.py --stem <stem>
